# AtmoSound - Data Preprocessing and Feature Engineering Pipeline

Pipeline Summary:
1. Drop fully-empty columns
2. Boolean attributes: treat missing as third "unknown" value
3. Rating: impute missing with median
4. Price level: ordinal encode 1-4, missing indicator column
5. Neighbourhood and primary_type: one-hot encode
6. review_summary and generative_summary: TF-IDF then truncated SVD (~50 dims)
7. Min-max normalize all features to [0, 1]
8. Spotify: clean, normalize tempo/loudness, compute per-genre profiles
9. Pseudo-label construction
10. Stratified train/val/test split (seed=42)


## Setup

In [330]:
import numpy as np
import pandas as pd
import os
import re
import pickle
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

PLACES_CSV = "dataset/manhattan_places.csv"
SPOTIFY_CSV = "dataset/songs_dataset.csv"

AUDIO_FEATURES = [
    "danceability", "energy", "acousticness", "valence",
    "instrumentalness", "liveness", "speechiness"
]

BOOL_COLS = [
    "good_for_children", "good_for_groups", "good_for_watching_sports",
    "allows_dogs", "live_music", "outdoor_seating", "reservable",
    "serves_beer", "serves_cocktails", "serves_wine", "serves_coffee",
    "serves_breakfast", "serves_brunch", "serves_dinner", "serves_lunch",
    "serves_vegetarian_food", "serves_dessert", "menu_for_children"
]

os.makedirs("pipeline_artifacts", exist_ok=True)
print("Setup complete.")

Setup complete.


## Load Raw Data

In [332]:
places_raw = pd.read_csv(PLACES_CSV, encoding="utf-8-sig")
print(f"Places raw: {places_raw.shape[0]:,} venues x {places_raw.shape[1]} columns")

spotify_raw = pd.read_csv(SPOTIFY_CSV)
print(f"Spotify raw: {spotify_raw.shape[0]:,} tracks x {spotify_raw.shape[1]} columns")

Places raw: 4,484 venues x 41 columns
Spotify raw: 114,000 tracks x 21 columns


In [333]:
print("Missing Values")
missing = places_raw.isnull().sum()
missing_pct = (missing / len(places_raw) * 100).round(1)
audit = pd.DataFrame({"missing": missing, "pct": missing_pct})
print(audit[audit["missing"] > 0].sort_values("pct", ascending=False).to_string())

Missing Values
                          missing    pct
attributions                 4484  100.0
neighborhood_summary         4484  100.0
pure_service_area            4484  100.0
allows_dogs                  3267   72.9
good_for_groups              3038   67.8
serves_brunch                2848   63.5
serves_breakfast             2688   59.9
serves_vegetarian_food       2641   58.9
serves_dinner                2507   55.9
reservable                   2473   55.2
menu_for_children            2456   54.8
outdoor_seating              2444   54.5
serves_lunch                 2431   54.2
serves_cocktails             2408   53.7
good_for_children            2376   53.0
serves_beer                  2317   51.7
serves_wine                  2312   51.6
good_for_watching_sports     2281   50.9
live_music                   2250   50.2
serves_dessert               2230   49.7
price_level                  2190   48.8
serves_coffee                2177   48.6
price_range                  2140   47.7
r

## Drop Fully-Empty Columns

In [335]:
COLS_TO_DROP = ["pure_service_area", "neighborhood_summary", "attributions"]
existing_drops = [c for c in COLS_TO_DROP if c in places_raw.columns]
places = places_raw.drop(columns=existing_drops).copy()
print(f"Dropped {len(existing_drops)} columns: {existing_drops}")
print(f"Shape after drop: {places.shape}")

Dropped 3 columns: ['pure_service_area', 'neighborhood_summary', 'attributions']
Shape after drop: (4484, 38)


## Boolean Attribute Handling

Encode: True = 1, False = 0, Missing = -1 (unknown).
Missingness correlates with venue type, so treating it as informative rather than imputing.

In [337]:
for col in BOOL_COLS:
    if col in places.columns:
        places[col] = places[col].map({True: 1, False: 0, 1: 1, 0: 0, 1.0: 1, 0.0: 0})
        places[col] = places[col].fillna(-1).astype(int)

print("Boolean coverage:")
for col in BOOL_COLS:
    if col in places.columns:
        known = (places[col] != -1).sum()
        pct = known / len(places) * 100
        print(f"  {col:35s}: {pct:5.1f}% known")

Boolean coverage:
  good_for_children                  :  47.0% known
  good_for_groups                    :  32.2% known
  good_for_watching_sports           :  49.1% known
  allows_dogs                        :  27.1% known
  live_music                         :  49.8% known
  outdoor_seating                    :  45.5% known
  reservable                         :  44.8% known
  serves_beer                        :  48.3% known
  serves_cocktails                   :  46.3% known
  serves_wine                        :  48.4% known
  serves_coffee                      :  51.4% known
  serves_breakfast                   :  40.1% known
  serves_brunch                      :  36.5% known
  serves_dinner                      :  44.1% known
  serves_lunch                       :  45.8% known
  serves_vegetarian_food             :  41.1% known
  serves_dessert                     :  50.3% known
  menu_for_children                  :  45.2% known


## Rating Imputation

Impute missing ratings with the median.

In [339]:
print(f"Missing before: {places['rating'].isnull().sum()}")
rating_median = places["rating"].median()
places["rating"] = places["rating"].fillna(rating_median)
print(f"Imputed with median: {rating_median}")
print(f"Missing after: {places['rating'].isnull().sum()}")

Missing before: 88
Imputed with median: 4.5
Missing after: 0


## Price Level Encoding

Ordinal encode 1-4 with a separate missing indicator column.

In [341]:
PRICE_MAP = {
    "PRICE_LEVEL_INEXPENSIVE": 1,
    "PRICE_LEVEL_MODERATE": 2,
    "PRICE_LEVEL_EXPENSIVE": 3,
    "PRICE_LEVEL_VERY_EXPENSIVE": 4,
    "Inexpensive": 1,
    "Moderate": 2,
    "Expensive": 3,
    "Very Expensive": 4,
}

places["price_level_missing"] = places["price_level"].isnull().astype(int)
places["price_level_encoded"] = places["price_level"].map(PRICE_MAP)

if places["price_level_encoded"].isnull().all():
    places["price_level_encoded"] = pd.to_numeric(places["price_level"], errors="coerce")

places["price_level_encoded"] = places["price_level_encoded"].fillna(0).astype(int)

print(f"Missing indicator sum: {places['price_level_missing'].sum()}")
print(f"Encoded distribution:\n{places['price_level_encoded'].value_counts().sort_index().to_string()}")

Missing indicator sum: 2190
Encoded distribution:
price_level_encoded
0    2190
1     435
2    1483
3     290
4      86


## One-Hot Encode Neighbourhood and Primary Type

Using pd.get_dummies (Pandas only).

In [343]:
neighbourhood_dummies = pd.get_dummies(places["neighbourhood"], prefix="neigh", dtype=int)
print(f"Neighbourhood one-hot: {neighbourhood_dummies.shape[1]} columns")

primarytype_dummies = pd.get_dummies(places["primary_type"], prefix="ptype", dtype=int)
print(f"Primary type one-hot: {primarytype_dummies.shape[1]} columns")

Neighbourhood one-hot: 33 columns
Primary type one-hot: 175 columns


## TF-IDF Vectorization

In [345]:
class TfidfVectorizerFromScratch:
    """TF-IDF vectorizer using only NumPy and Python stdlib."""

    def __init__(self, max_features=5000, ngram_range=(1, 2), sublinear_tf=True,
                 min_df=2, max_df_ratio=0.95):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.sublinear_tf = sublinear_tf
        self.min_df = min_df
        self.max_df_ratio = max_df_ratio
        self.vocabulary_ = None
        self.idf_ = None

    def _tokenize(self, text):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return []
        text = re.sub(r"[^a-z\s]", " ", text.lower())
        return [t for t in text.split() if len(t) > 1]

    def _get_ngrams(self, tokens):
        ngrams = []
        for n in range(self.ngram_range[0], self.ngram_range[1] + 1):
            for i in range(len(tokens) - n + 1):
                ngrams.append(" ".join(tokens[i:i+n]))
        return ngrams

    def fit(self, documents):
        n_docs = len(documents)
        max_df_count = int(self.max_df_ratio * n_docs)

        doc_freq = Counter()
        for doc in documents:
            ngrams = set(self._get_ngrams(self._tokenize(doc)))
            doc_freq.update(ngrams)

        valid_terms = {
            term: df for term, df in doc_freq.items()
            if df >= self.min_df and df <= max_df_count
        }

        sorted_terms = sorted(valid_terms.keys(), key=lambda t: valid_terms[t], reverse=True)
        if self.max_features and len(sorted_terms) > self.max_features:
            sorted_terms = sorted_terms[:self.max_features]

        self.vocabulary_ = {term: idx for idx, term in enumerate(sorted(sorted_terms))}

        self.idf_ = np.zeros(len(self.vocabulary_))
        for term, idx in self.vocabulary_.items():
            self.idf_[idx] = np.log(n_docs / (1 + valid_terms[term])) + 1

        return self

    def transform(self, documents):
        n_docs = len(documents)
        n_features = len(self.vocabulary_)
        tfidf_matrix = np.zeros((n_docs, n_features))

        for i, doc in enumerate(documents):
            ngrams = self._get_ngrams(self._tokenize(doc))
            if len(ngrams) == 0:
                continue
            term_counts = Counter(ngrams)
            for term, count in term_counts.items():
                if term in self.vocabulary_:
                    idx = self.vocabulary_[term]
                    tf = count / len(ngrams)
                    if self.sublinear_tf:
                        tf = 1 + np.log(tf) if tf > 0 else 0
                    tfidf_matrix[i, idx] = tf * self.idf_[idx]

        row_norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
        row_norms[row_norms == 0] = 1
        tfidf_matrix = tfidf_matrix / row_norms
        return tfidf_matrix

    def fit_transform(self, documents):
        return self.fit(documents).transform(documents)

print("TfidfVectorizerFromScratch defined.")

TfidfVectorizerFromScratch defined.


In [346]:
places["review_summary"] = places["review_summary"].fillna("")
places["generative_summary"] = places["generative_summary"].fillna("")
places["combined_text"] = (places["review_summary"] + " " + places["generative_summary"]).str.strip()

has_text = (places["combined_text"].str.len() > 0).sum()
print(f"Venues with text: {has_text}/{len(places)} ({has_text/len(places)*100:.1f}%)")

tfidf = TfidfVectorizerFromScratch(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_df_ratio=0.95)
tfidf_matrix = tfidf.fit_transform(places["combined_text"].tolist())

print(f"TF-IDF matrix: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

Venues with text: 3534/4484 (78.8%)
TF-IDF matrix: (4484, 5000)
Vocabulary size: 5000


## Truncated SVD

Reduce TF-IDF to 50 dimensions using np.linalg.svd.

In [348]:
class TruncatedSVDFromScratch:
    """Truncated SVD using np.linalg.svd. Keeps top n_components."""

    def __init__(self, n_components=50):
        self.n_components = n_components
        self.components_ = None
        self.singular_values_ = None
        self.mean_ = None

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        self.components_ = Vt[:self.n_components]
        self.singular_values_ = S[:self.n_components]
        total_var = np.sum(S ** 2)
        self.explained_variance_ratio_ = np.sum(self.singular_values_ ** 2) / total_var
        return self

    def transform(self, X):
        return (X - self.mean_) @ self.components_.T

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

N_SVD_COMPONENTS = 50
svd = TruncatedSVDFromScratch(n_components=N_SVD_COMPONENTS)
text_svd = svd.fit_transform(tfidf_matrix)

print(f"SVD: {tfidf_matrix.shape[1]} -> {text_svd.shape[1]} dimensions")
print(f"Variance explained: {svd.explained_variance_ratio_*100:.1f}%")

SVD: 5000 -> 50 dimensions
Variance explained: 19.9%


## Assemble Feature Matrix

In [350]:
feature_groups = {}

feature_groups["rating"] = places[["rating"]].values
feature_groups["price"] = places[["price_level_encoded", "price_level_missing"]].values
feature_groups["booleans"] = places[[c for c in BOOL_COLS if c in places.columns]].values
feature_groups["neighbourhood"] = neighbourhood_dummies.values
feature_groups["primary_type"] = primarytype_dummies.values
feature_groups["text_svd"] = text_svd

for k, v in feature_groups.items():
    print(f"  {k:20s}: {v.shape[1]} dims")

X_all = np.hstack([feature_groups[k] for k in feature_groups])

feature_names = (
    ["rating"] +
    ["price_level_encoded", "price_level_missing"] +
    [c for c in BOOL_COLS if c in places.columns] +
    list(neighbourhood_dummies.columns) +
    list(primarytype_dummies.columns) +
    [f"text_svd_{i}" for i in range(N_SVD_COMPONENTS)]
)

print(f"\nTotal feature matrix: {X_all.shape}")
assert len(feature_names) == X_all.shape[1], "Feature name count mismatch!"

  rating              : 1 dims
  price               : 2 dims
  booleans            : 18 dims
  neighbourhood       : 33 dims
  primary_type        : 175 dims
  text_svd            : 50 dims

Total feature matrix: (4484, 279)


## Min-Max Normalize to [0, 1]

In [352]:
class MinMaxScalerFromScratch:
    """Min-max scaler using only NumPy."""

    def __init__(self):
        self.min_ = None
        self.range_ = None

    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.range_ = X.max(axis=0) - self.min_
        self.range_[self.range_ == 0] = 1.0
        return self

    def transform(self, X):
        return (X - self.min_) / self.range_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

scaler = MinMaxScalerFromScratch()
X_scaled = scaler.fit_transform(X_all)

print(f"Normalized range: [{X_scaled.min():.4f}, {X_scaled.max():.4f}]")
print(f"Shape: {X_scaled.shape}")

Normalized range: [0.0000, 1.0000]
Shape: (4484, 279)


## Spotify dataset Preprocessing

Drop missing names, normalize tempo and loudness, filter low-popularity, compute per-genre profiles.

In [354]:
spotify = spotify_raw.dropna(subset=["artists", "track_name"]).copy()
spotify.reset_index(drop=True, inplace=True)
print(f"After dropping missing names: {spotify.shape[0]:,} tracks")

spotify["tempo_norm"] = spotify["tempo"].clip(0, 243) / 243.0
spotify["loudness_norm"] = (spotify["loudness"].clip(-60, 0) + 60) / 60.0

MIN_POPULARITY = 10
spotify_filtered = spotify[spotify["popularity"] >= MIN_POPULARITY].copy()
spotify_filtered.reset_index(drop=True, inplace=True)
print(f"After popularity filter (>= {MIN_POPULARITY}): {spotify_filtered.shape[0]:,} tracks")
print(f"Genres: {spotify_filtered['track_genre'].nunique()}")

After dropping missing names: 113,999 tracks
After popularity filter (>= 10): 91,271 tracks
Genres: 114


In [355]:
genre_profiles = spotify_filtered.groupby("track_genre")[AUDIO_FEATURES].mean()
print(f"Genre profiles: {genre_profiles.shape[0]} genres x {genre_profiles.shape[1]} audio dims")

MERGE_MAP = {
    "singer-songwriter": "songwriter",
    "latino": "reggaeton",
}

spotify_filtered["track_genre_clean"] = spotify_filtered["track_genre"].replace(MERGE_MAP)
genre_profiles_merged = spotify_filtered.groupby("track_genre_clean")[AUDIO_FEATURES].mean()
print(f"After merging near-duplicates: {genre_profiles_merged.shape[0]} genres")

Genre profiles: 114 genres x 7 audio dims
After merging near-duplicates: 112 genres


## Ground Truth Label Construction (Pseudo-Labels)

We assign each venue a 7-dimensional target audio profile using its structured attributes.

Steps:
1. Map each venue's primary_type to a set of genre names via substring matching
2. Look up those genres' mean audio profiles from genre_profiles_merged (computed from Spotify data)
3. Average the matched genre profiles to get a base audio profile for the venue
4. Modulate the base profile using boolean attribute flags (e.g. live_music shifts toward jazz/blues/rock, serves_cocktails shifts toward jazz/soul/chill)
5. Apply price-level adjustments (higher price shifts toward more acoustic/instrumental)
6. Venues with no type match fall back to the global genre mean

The genre-to-audio-feature values are entirely data-driven from the Spotify dataset.
All labels are constructed before splitting to prevent leakage.

In [357]:
# Build a primary_type to audio profile mapping using genre associations.
# This required a bit of manual intervention or would not work/similarity scores were not worth interpreting
# so we map venue types to genre groups here
# The actual audio VALUES come from genre_profiles_merged.

TYPE_TO_GENRES = {
    # Restaurants
    "restaurant":        ["jazz", "pop", "soul", "indie", "acoustic"],
    "italian":           ["jazz", "classical", "soul", "acoustic"],
    "mexican":           ["latin", "reggaeton", "pop", "rock"],
    "japanese":          ["ambient", "jazz", "classical", "chill"],
    "chinese":           ["ambient", "jazz", "pop", "classical"],
    "indian":            ["world-music", "ambient", "jazz", "soul"],
    "thai":              ["ambient", "chill", "jazz", "world-music"],
    "french":            ["jazz", "classical", "french", "acoustic"],
    "american":          ["rock", "pop", "country", "blues", "soul"],
    "seafood":           ["jazz", "acoustic", "ambient", "soul"],
    "steak":             ["jazz", "blues", "soul", "classical"],
    "mediterranean":     ["jazz", "world-music", "ambient", "soul"],
    "pizza":             ["pop", "rock", "indie"],
    "korean":            ["k-pop", "pop", "ambient", "chill"],
    "vietnamese":        ["ambient", "chill", "jazz"],
    "turkish":           ["world-music", "ambient", "jazz"],
    "spanish":           ["latin", "jazz", "classical"],
    "sushi":             ["jazz", "ambient", "classical", "chill"],
    "ramen":             ["pop", "ambient", "rock", "indie"],
    "vegan":             ["indie", "folk", "acoustic", "ambient"],
    "vegetarian":        ["indie", "folk", "acoustic"],
    "breakfast":         ["acoustic", "folk", "indie", "jazz"],
    "brunch":            ["indie", "acoustic", "pop", "jazz"],
    "fast_food":         ["pop", "hip-hop", "electronic"],
    "sandwich":          ["indie", "pop", "acoustic", "folk"],
    "bakery":            ["acoustic", "jazz", "classical", "folk"],
    "dessert":           ["pop", "jazz", "acoustic", "indie"],
    # Cafes
    "cafe":              ["acoustic", "jazz", "indie", "folk"],
    "coffee":            ["acoustic", "indie", "jazz", "folk", "chill"],
    "tea":               ["ambient", "classical", "jazz", "chill"],
    # Bars and nightlife
    "bar":               ["rock", "pop", "indie", "hip-hop", "electronic"],
    "grill":             ["rock", "pop", "country", "blues"],
    "night_club":        ["electronic", "dance", "hip-hop", "pop", "house"],
    "lounge":            ["jazz", "r-n-b", "soul", "chill", "ambient"],
    # Retail
    "clothing":          ["pop", "electronic", "indie", "hip-hop"],
    "shoe":              ["pop", "hip-hop", "electronic", "r-n-b"],
    "jewelry":           ["jazz", "classical", "ambient"],
    "book":              ["acoustic", "classical", "ambient", "folk"],
    "shopping":          ["pop", "electronic", "dance", "hip-hop"],
    "gift":              ["indie", "pop", "acoustic"],
    "furniture":         ["ambient", "chill", "jazz", "acoustic"],
    "home":              ["ambient", "indie", "pop", "acoustic"],
    "sporting":          ["rock", "electronic", "hip-hop", "pop"],
    "beauty":            ["pop", "r-n-b", "soul", "jazz"],
    "hair":              ["pop", "r-n-b", "hip-hop", "indie"],
    # Fitness
    "gym":               ["electronic", "hip-hop", "rock", "pop", "dance"],
    "fitness":           ["electronic", "hip-hop", "dance", "pop"],
    "yoga":              ["ambient", "classical", "acoustic", "chill"],
    "sports":            ["rock", "electronic", "hip-hop", "pop"],
    # Hospitality
    "hotel":             ["jazz", "ambient", "classical", "chill"],
    "resort":            ["chill", "ambient", "jazz", "reggae"],
    "spa":               ["ambient", "classical", "acoustic", "chill"],
    # Other
    "museum":            ["classical", "ambient", "jazz"],
    "gallery":           ["ambient", "classical", "jazz", "electronic"],
    "theater":           ["pop", "rock", "indie", "electronic"],
    "bowling":           ["rock", "pop", "hip-hop", "indie"],
    "pharmacy":          ["ambient", "pop", "acoustic"],
    "supermarket":       ["pop", "acoustic", "indie"],
}

def get_type_profile(primary_type):
    """Match primary_type against TYPE_TO_GENRES using substring matching."""
    if not isinstance(primary_type, str):
        return None
    pt = primary_type.lower().replace("_", " ")
    # Try each key as a substring match against the primary_type
    best_match = None
    best_len = 0
    for key, genres in TYPE_TO_GENRES.items():
        if key in pt and len(key) > best_len:
            best_match = genres
            best_len = len(key)
    if best_match is None:
        return None
    # Look up audio profiles for matched genres
    matched = [genre_profiles_merged.loc[g].values for g in best_match if g in genre_profiles_merged.index]
    if matched:
        return np.mean(matched, axis=0)
    return None

# Test coverage
covered = sum(1 for pt in places["primary_type"] if get_type_profile(pt) is not None)
print(f"Primary types with profile: {covered}/{len(places)} ({covered/len(places)*100:.1f}%)")

Primary types with profile: 4076/4484 (90.9%)


In [358]:
# Build targets using: primary_type base profile + boolean attribute modulation
# Boolean modulation is data-driven: each flag shifts the profile toward
# genres associated with that attribute.

BOOL_GENRE_SHIFTS = {
    "live_music":               (["jazz", "blues", "rock", "folk", "acoustic", "soul"], +0.15),
    "good_for_watching_sports": (["rock", "hip-hop", "electronic", "pop"], +0.12),
    "outdoor_seating":          (["folk", "acoustic", "indie", "reggae"], +0.08),
    "serves_cocktails":         (["jazz", "soul", "r-n-b", "chill"], +0.08),
    "serves_beer":              (["rock", "country", "blues", "indie"], +0.06),
    "serves_wine":              (["jazz", "classical", "acoustic"], +0.06),
    "serves_coffee":            (["acoustic", "indie", "jazz", "folk"], +0.06),
    "good_for_children":        (["pop", "acoustic", "indie", "folk"], +0.04),
    "allows_dogs":              (["indie", "folk", "acoustic"], +0.04),
}

# Precompute boolean shift profiles
bool_shift_profiles = {}
for flag, (genres, weight) in BOOL_GENRE_SHIFTS.items():
    matched = [genre_profiles_merged.loc[g].values for g in genres if g in genre_profiles_merged.index]
    if matched:
        bool_shift_profiles[flag] = (np.mean(matched, axis=0), weight)

# Price level shifts: higher price -> more acoustic/instrumental
PRICE_SHIFTS = {
    1: np.array([+0.03, +0.03, -0.02, +0.02, -0.01, 0, 0]),
    2: np.array([0, 0, 0, 0, 0, 0, 0]),
    3: np.array([-0.02, -0.02, +0.04, 0, +0.03, 0, 0]),
    4: np.array([-0.03, -0.04, +0.06, 0, +0.05, 0, 0]),
}

y_targets_list = []
match_counts = {"typed": 0, "fallback": 0}
global_mean = genre_profiles_merged.mean().values

for i in range(len(places)):
    ptype = places["primary_type"].iloc[i]
    base = get_type_profile(ptype)

    if base is None:
        y_targets_list.append(global_mean.copy())
        match_counts["fallback"] += 1
        continue

    profile = base.copy()
    match_counts["typed"] += 1

    # Apply boolean shifts
    for flag, (shift_profile, weight) in bool_shift_profiles.items():
        if flag in places.columns:
            val = places[flag].iloc[i]
            if val == 1:
                profile = profile * (1 - weight) + shift_profile * weight
            # val == -1 (unknown) or val == 0: no change

    # Apply price shift
    price = places["price_level_encoded"].iloc[i]
    if price in PRICE_SHIFTS:
        profile = profile + PRICE_SHIFTS[price]

    profile = np.clip(profile, 0, 1)
    y_targets_list.append(profile)

y_targets = np.array(y_targets_list)

print(f"Target matrix: {y_targets.shape}")
print(f"Range: [{y_targets.min():.4f}, {y_targets.max():.4f}]")
print(f"Matched by type: {match_counts['typed']}")
print(f"Fallback (global mean): {match_counts['fallback']}")
print(f"\nMean profile across all venues:")
for feat, val in zip(AUDIO_FEATURES, y_targets.mean(axis=0)):
    print(f"  {feat:20s}: {val:.3f}")

Target matrix: (4484, 7)
Range: [0.0305, 0.7622]
Matched by type: 4076
Fallback (global mean): 408

Mean profile across all venues:
  danceability        : 0.597
  energy              : 0.549
  acousticness        : 0.421
  valence             : 0.460
  instrumentalness    : 0.099
  liveness            : 0.173
  speechiness         : 0.073


In [359]:
# Inspect sample venues
sample_indices = np.random.RandomState(SEED).choice(len(places), size=8, replace=False)

for idx in sample_indices:
    ptype = places["primary_type"].iloc[idx]
    price = places["price_level_encoded"].iloc[idx]
    bools_on = [c for c in BOOL_GENRE_SHIFTS if c in places.columns and places[c].iloc[idx] == 1]
    print(f"\nVenue: {ptype} (price={price})")
    print(f"  Active flags: {bools_on if bools_on else 'none'}")
    print(f"  Target: {dict(zip(AUDIO_FEATURES, y_targets[idx].round(3)))}")


Venue: bar (price=0)
  Active flags: ['live_music', 'outdoor_seating', 'serves_cocktails', 'serves_beer', 'serves_wine', 'serves_coffee']
  Target: {'danceability': 0.608, 'energy': 0.583, 'acousticness': 0.36, 'valence': 0.472, 'instrumentalness': 0.074, 'liveness': 0.175, 'speechiness': 0.074}

Venue: sportswear_store (price=0)
  Active flags: none
  Target: {'danceability': 0.637, 'energy': 0.662, 'acousticness': 0.248, 'valence': 0.479, 'instrumentalness': 0.072, 'liveness': 0.178, 'speechiness': 0.087}

Venue: italian_restaurant (price=2)
  Active flags: ['serves_cocktails', 'serves_beer', 'serves_wine', 'serves_coffee', 'good_for_children']
  Target: {'danceability': 0.588, 'energy': 0.5, 'acousticness': 0.482, 'valence': 0.457, 'instrumentalness': 0.066, 'liveness': 0.164, 'speechiness': 0.062}

Venue: american_restaurant (price=2)
  Active flags: ['serves_cocktails', 'serves_beer', 'serves_wine', 'serves_coffee']
  Target: {'danceability': 0.588, 'energy': 0.498, 'acousticness

## Stratified Train / Validation / Test Split

80/20 train/test stratified by primary_type, training further split 80/20 train/val. Seed=42.

In [361]:
def stratified_split(X, y, groups, test_ratio=0.2, seed=42):
    rng = np.random.RandomState(seed)
    unique_groups = np.unique(groups)
    idx_a, idx_b = [], []

    for group in unique_groups:
        group_indices = np.where(groups == group)[0]
        rng.shuffle(group_indices)
        n_test = max(1, int(len(group_indices) * test_ratio))
        idx_b.extend(group_indices[:n_test])
        idx_a.extend(group_indices[n_test:])

    idx_a = np.array(idx_a)
    idx_b = np.array(idx_b)
    return X[idx_a], y[idx_a], idx_a, X[idx_b], y[idx_b], idx_b

places["primary_type"] = places["primary_type"].fillna("unknown")
strat_groups = places["primary_type"].values

X_train_full, y_train_full, idx_train_full, X_test, y_test, idx_test = stratified_split(
    X_scaled, y_targets, strat_groups, test_ratio=0.2, seed=SEED
)

strat_groups_train = strat_groups[idx_train_full]
X_train, y_train, _, X_val, y_val, _ = stratified_split(
    X_train_full, y_train_full, strat_groups_train, test_ratio=0.2, seed=SEED
)

print(f"Train:      {X_train.shape[0]:,} samples")
print(f"Validation: {X_val.shape[0]:,} samples")
print(f"Test:       {X_test.shape[0]:,} samples")
print(f"Features:   {X_train.shape[1]}")
print(f"Targets:    {y_train.shape[1]}")

Train:      2,853 samples
Validation: 714 samples
Test:       917 samples
Features:   279
Targets:    7


## Baselines

Mean-prediction and random-genre baselines. Models must beat these.

In [363]:
y_mean_pred = np.tile(y_train.mean(axis=0), (X_test.shape[0], 1))
mse_mean = np.mean((y_test - y_mean_pred) ** 2)

rng = np.random.RandomState(SEED)
random_genre_indices = rng.choice(len(genre_profiles_merged), size=X_test.shape[0])
y_random_pred = genre_profiles_merged.values[random_genre_indices]
mse_random = np.mean((y_test - y_random_pred) ** 2)

print(f"Mean-prediction MSE: {mse_mean:.4f}")
print(f"Random-genre MSE:    {mse_random:.4f}")
print(f"Models must achieve MSE < {mse_mean:.4f} to beat mean baseline.")

Mean-prediction MSE: 0.0041
Random-genre MSE:    0.0319
Models must achieve MSE < 0.0041 to beat mean baseline.


## Save Artifacts

In [373]:
os.makedirs("pipeline_artifacts", exist_ok=True)
np.savez(
    "pipeline_artifacts/model_ready_data.npz",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
    feature_names=np.array(feature_names),
    audio_feature_names=np.array(AUDIO_FEATURES),
)
print("Saved: pipeline_artifacts/model_ready_data.npz")

genre_profiles_merged.to_csv("pipeline_artifacts/genre_profiles.csv")
print("Saved: pipeline_artifacts/genre_profiles.csv")

deployment_artifacts = {
    "tfidf_vocabulary": tfidf.vocabulary_,
    "tfidf_idf": tfidf.idf_,
    "tfidf_config": {
        "max_features": tfidf.max_features,
        "ngram_range": tfidf.ngram_range,
        "sublinear_tf": tfidf.sublinear_tf,
        "min_df": tfidf.min_df,
        "max_df_ratio": tfidf.max_df_ratio,
    },
    "svd_components": svd.components_,
    "svd_mean": svd.mean_,
    "svd_singular_values": svd.singular_values_,
    "scaler_min": scaler.min_,
    "scaler_range": scaler.range_,
    "feature_names": feature_names,
    "bool_cols": BOOL_COLS,
    "audio_features": AUDIO_FEATURES,
    "merge_map": MERGE_MAP,
    "rating_median": rating_median,
}
with open("pipeline_artifacts/deployment_transformers.pkl", "wb") as f:
    pickle.dump(deployment_artifacts, f)
print("Saved: pipeline_artifacts/deployment_transformers.pkl")

spotify_filtered.to_csv("pipeline_artifacts/spotify_filtered.csv", index=False)
print("Saved: pipeline_artifacts/spotify_filtered.csv")

Saved: pipeline_artifacts/model_ready_data.npz
Saved: pipeline_artifacts/genre_profiles.csv
Saved: pipeline_artifacts/deployment_transformers.pkl
Saved: pipeline_artifacts/spotify_filtered.csv


## Handoff Notes

For ML algorithms (Ridge and Neural Network):
```python
data = np.load("pipeline_artifacts/model_ready_data.npz", allow_pickle=True)
X_train = data["X_train"]
y_train = data["y_train"]
X_val   = data["X_val"]
y_val   = data["y_val"]
X_test  = data["X_test"]
y_test  = data["y_test"]
feature_names = data["feature_names"]
```

For evaluation (ablation feature indices):
```python
bool_idx = [i for i, c in enumerate(feature_names) if c in BOOL_COLS]
text_idx = [i for i, c in enumerate(feature_names) if c.startswith("text_svd_")]
```

For Streamlit deployment:
```python
with open("pipeline_artifacts/deployment_transformers.pkl", "rb") as f:
    artifacts = pickle.load(f)

# Reconstruct TF-IDF vectorizer
tfidf = TfidfVectorizerFromScratch(**artifacts["tfidf_config"])
tfidf.vocabulary_ = artifacts["tfidf_vocabulary"]
tfidf.idf_ = artifacts["tfidf_idf"]

# Reconstruct SVD transformer
svd = TruncatedSVDFromScratch(n_components=artifacts["svd_components"].shape[0])
svd.components_ = artifacts["svd_components"]
svd.mean_ = artifacts["svd_mean"]
svd.singular_values_ = artifacts["svd_singular_values"]

# Reconstruct scaler
scaler = MinMaxScalerFromScratch()
scaler.min_ = artifacts["scaler_min"]
scaler.range_ = artifacts["scaler_range"]

# Then call tfidf.transform(), svd.transform(),scaler.transform() on new venue data
```
